# C1EÑ en Kaggle: funciones de Ñ en las dos T4

**C1EÑ** («Cuda 1 Engine of Ñ») compila funciones escritas en un subconjunto numérico de Ñ a tres destinos —JavaScript (`jit`), C con gcc (`nativo`) y CUDA C con NVRTC (`gpu`)— y promete **los mismos bits** en los tres para `+ − * /` y `raiz`.

Este cuaderno:
1. instala Node y koffi si hace falta y consigue el proyecto (dataset subido por ti **o** `git clone`);
2. enseña las GPU (`nvidia-smi`);
3. corre la suite de C1EÑ **contra la GPU de verdad**, comparando GPU, jit y nativo bit a bit (incluidos 300 núcleos inventados al azar);
4. mide un Monte Carlo de π, un mandelbrot y un n-cuerpos en jit / nativo / gpu y los pone en una tabla;
5. reparte un trabajo entre **las dos T4** con un proceso por GPU (`C1EN_GPU=0` y `C1EN_GPU=1`).

**Antes de empezar:** *Settings → Accelerator → GPU T4 x2*. Si vas a usar la opción B (clonar de GitHub), también *Settings → Internet → On*.

In [ ]:
# 1 · Node 22 (Kaggle trae uno viejo o ninguno) — se instala en /kaggle/temp (no va al Output) sin tocar el sistema
import os, subprocess, shutil, re
import glob
os.makedirs('/kaggle/temp', exist_ok=True)   # lo pesado (Node, el motor de Ñ) va aquí: Kaggle no lo guarda en el Output
for v in [f'/kaggle/working/{x}' for x in ['n', 'bench-c1en.esl', 'pi-trozo.esl']] + glob.glob('/kaggle/working/node-v*-linux-x64'):
    if os.path.isdir(v) and not os.path.islink(v): shutil.rmtree(v, ignore_errors=True); print('borrado del Output:', v)
    elif os.path.exists(v): os.remove(v); print('borrado del Output:', v)
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /kaggle/temp', shell=True, check=True)
    os.environ['PATH'] = f'/kaggle/temp/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())

In [ ]:
# 2 · El proyecto. Opción A: un dataset de Kaggle con la carpeta del proyecto (la que tiene motor/ dentro).
#     Opción B: git clone del repositorio. Se usa A si se encuentra; si no, B.
import glob
PROYECTO = '/kaggle/temp/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO):
    print('ya estaba en', PROYECTO)
elif candidatos:
    # /kaggle/input es de solo lectura y hay que compilar el bundle e instalar koffi: se copia.
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules'))
    print('opción A: copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
    print('opción B: clonado en', PROYECTO)
os.chdir(PROYECTO)
assert os.path.exists('motor/src/14-c1en.js'), 'este proyecto no trae C1EÑ (motor/src/14-c1en.js): ¿es una versión vieja?'

In [ ]:
# 3 · koffi (el FFI de Node que habla con libcuda y libnvrtc) y el bundle del motor
!cd /kaggle/temp/n && npm install koffi --no-audit --no-fund --loglevel=error && node motor/util/construir.js

In [ ]:
# 4 · Las GPU y el toolkit. C1EÑ busca libnvrtc solo; si no la encuentra, indícala con C1EN_NVRTC_LIB.
!nvidia-smi
!ls /usr/local/cuda/lib64/ 2>/dev/null | grep -E '^libnvrtc\.so' || echo 'no hay libnvrtc en /usr/local/cuda/lib64'
!cd /kaggle/temp/n && node -e "require('./motor/bundle.js'); const m = globalThis.EspanolLike.crearMotor({}); console.log(m.vm.c1en.disponibles().join('\n'))"

## La suite de C1EÑ contra la GPU de verdad

`--gpu` hace que la suite use la GPU real en vez del simulador. Compara **GPU, jit y nativo** en cada caso: el subconjunto, los rechazos, los fallos en ejecución (división entre cero, índices…: el hilo más bajo y la línea, igual en los tres), `azarHilo`, vistas no contiguas, reducciones por árbol, núcleos de biblioteca y **300 núcleos al azar** comparados al bit (los NaN se comparan como NaN, porque su carga útil no es la misma en la GPU).

La primera vez tarda: cada núcleo se compila con gcc y con NVRTC.

In [ ]:
!cd /kaggle/temp/n && node motor/pruebas/c1en.js --gpu --fuzz=300

In [ ]:
# La suite escrita en Ñ: con GPU, compara también "gpu" contra "jit" (saxpy, histograma, π, mandelbrot…)
!cd /kaggle/temp/n && node motor/util/n-node.js motor/pruebas/en-n/nen-c1en.esl | tail -40

## Benchmark: jit · nativo · gpu

Tres núcleos de Ñ. Cada uno se calienta una vez (compila) y se mide la mejor de tres. La columna «igual» dice si el resultado es **idéntico** al de `jit` (con `==` de Ñ, elemento a elemento).

In [ ]:
%%writefile /kaggle/temp/bench-c1en.esl
usar paralelo, dispositivos, usarDispositivo, azarHilo de "nen"
usar ceros, suma, raiz, aListaA de "numerico"

fn piHilo(i, muestras, semilla, dentro) {
    var c = 0
    para j en rango(muestras) {
        var x = azarHilo(semilla, i * muestras + j, 0)
        var y = azarHilo(semilla, i * muestras + j, 1)
        si x * x + y * y < 1.0 { c = c + 1 }
    }
    dentro[i] = c
}
fn mandel(i, ancho, alto, iter, r) {
    var cx = -2.0 + 3.0 * (i % ancho) / ancho
    var cy = -1.2 + 2.4 * (i // ancho) / alto
    var zx = 0.0
    var zy = 0.0
    var k = 0
    mientras k < iter y zx * zx + zy * zy < 4.0 {
        var t = zx * zx - zy * zy + cx
        zy = 2.0 * zx * zy + cy
        zx = t
        k = k + 1
    }
    r[i] = k
}
fn cuerpos(i, x, y, ax, ay) {
    var sx = 0.0
    var sy = 0.0
    para j en rango(longitud(x)) {
        var dx = x[j] - x[i]
        var dy = y[j] - y[i]
        var d2 = dx * dx + dy * dy + 0.01
        var inv = 1.0 / (d2 * raiz(d2))
        sx = sx + dx * inv
        sy = sy + dy * inv
    }
    ax[i] = sx
    ay[i] = sy
}

fn corto(d: texto) -> texto {
    si contiene(d, "gpu") { devolver "gpu" }
    si contiene(d, "nativo") { devolver "nativo" }
    devolver "jit"
}
# Una vez para calentar (compilar) y tres medidas; devuelve la mejor.
fn medir(nombre: texto, d: texto, f, n: entero, args) -> lista {
    var mejor = 1.0e300
    var res: cualquiera = nulo
    para k en rango(4) {
        var a = args()
        var t0 = reloj()
        f(n, a)
        var t = reloj() - t0
        si k > 0 y t < mejor { mejor = t }
        res = a
    }
    devolver [mejor, res]
}

fijo HILOS_PI = 65536
fijo MUESTRAS = 1000
fijo ANCHO = 1200
fijo ALTO = 800
fijo ITER = 500
fijo N_CUERPOS = 8192

var refs = {}
var vistos = []
para d0 en dispositivos() {
    var d = corto(d0)
    # con dos GPU aparecen "gpu 0" y "gpu 1": se mide una (la de C1EN_GPU)
    si contiene(vistos, d) { continuar }
    vistos = vistos + [d]
    usarDispositivo(d)
    # π: 65 536 hilos × 1000 muestras
    var mp = medir("pi", d, fn(n, a) { paralelo(n, piHilo, MUESTRAS, 2026, a[0]) }, HILOS_PI,
                   fn() { devolver [ceros(HILOS_PI, "entero")] })
    var pi = 4.0 * suma(mp[1][0]) / (HILOS_PI * MUESTRAS)
    # mandelbrot 1200×800, hasta 500 iteraciones
    var mm = medir("mandel", d, fn(n, a) { paralelo(n, mandel, ANCHO, ALTO, ITER, a[0]) }, ANCHO * ALTO,
                   fn() { devolver [ceros(ANCHO * ALTO, "entero")] })
    # n-cuerpos: 8192 cuerpos, todas las parejas
    var mc = medir("cuerpos", d, fn(n, a) { paralelo(n, cuerpos, a[0], a[1], a[2], a[3]) }, N_CUERPOS, fn() {
        var x = ceros(N_CUERPOS)
        var y = ceros(N_CUERPOS)
        paralelo(N_CUERPOS, fn(i, x, y) { x[i] = azarHilo(5, i, 0) * 10.0
                                          y[i] = azarHilo(5, i, 1) * 10.0 }, x, y)
        devolver [x, y, ceros(N_CUERPOS), ceros(N_CUERPOS)]
    })
    var r = {"pi": aListaA(mp[1][0]), "mandel": aListaA(mm[1][0]), "cuerpos": aListaA(mc[1][2])}
    si d == "jit" { refs = r }
    imprimir("RES|pi|" + d + "|" + texto(mp[0]) + "|" + texto(r["pi"] == refs["pi"]) + "|" + texto(pi))
    imprimir("RES|mandelbrot|" + d + "|" + texto(mm[0]) + "|" + texto(r["mandel"] == refs["mandel"]) + "|" + texto(suma(mm[1][0])))
    imprimir("RES|n-cuerpos|" + d + "|" + texto(mc[0]) + "|" + texto(r["cuerpos"] == refs["cuerpos"]) + "|" + texto(suma(mc[1][2])))
}

In [ ]:
import subprocess, pandas as pd
sal = subprocess.run(['node', 'motor/util/n-node.js', '/kaggle/temp/bench-c1en.esl'], capture_output=True, text=True)
if sal.returncode: print(sal.stdout, sal.stderr)
filas = [l.split('|')[1:] for l in sal.stdout.splitlines() if l.startswith('RES|')]
df = pd.DataFrame(filas, columns=['núcleo', 'dispositivo', 'ms', 'igual a jit', 'resultado'])
df['ms'] = df['ms'].astype(float)
tabla = df.pivot(index='núcleo', columns='dispositivo', values='ms')
for d in ['nativo', 'gpu']:
    if d in tabla: tabla[f'{d} × jit'] = (tabla['jit'] / tabla[d]).round(1)
print(tabla.round(1).to_string())
print()
print(df[['núcleo', 'dispositivo', 'igual a jit', 'resultado']].to_string(index=False))

## Las dos T4: un proceso por GPU

El patrón recomendado: **un proceso por GPU**, cada uno con su `C1EN_GPU`. Cada proceso tiene su contexto CUDA y su caché de módulos, y no hay que pensar en qué GPU está activa. Aquí se parte el Monte Carlo de π en dos mitades de hilos; como `azarHilo` es una función pura de `(semilla, i, canal)`, **la suma de las dos mitades es exactamente la cuenta de una sola GPU**.

(Dentro de un mismo proceso también vale `usarDispositivo("gpu", 1)`, pero los lanzamientos van uno detrás de otro.)

In [ ]:
%%writefile /kaggle/temp/pi-trozo.esl
usar paralelo, usarDispositivo, dispositivo, azarHilo de "nen"
usar ceros, suma de "numerico"
# argumentos: primer hilo, cuántos hilos, muestras por hilo
fn piHilo(i, desde, muestras, dentro) {
    var c = 0
    para j en rango(muestras) {
        var x = azarHilo(2026, (i + desde) * muestras + j, 0)
        var y = azarHilo(2026, (i + desde) * muestras + j, 1)
        si x * x + y * y < 1.0 { c = c + 1 }
    }
    dentro[i] = c
}
var a = argumentos()
var desde = entero(a[0])
var hilos = entero(a[1])
var muestras = entero(a[2])
usarDispositivo("gpu")
var dentro = ceros(hilos, "entero")
paralelo(1, piHilo, desde, 1, ceros(1, "entero"))      # calienta: compila con NVRTC
var t0 = reloj()
paralelo(hilos, piHilo, desde, muestras, dentro)
imprimir(dispositivo() + "|" + texto(suma(dentro)) + "|" + texto(reloj() - t0))

In [ ]:
import subprocess, time, os
HILOS, MUESTRAS = 1 << 20, 2000          # 2 100 millones de puntos
def lanzar(gpu, desde, hilos):
    env = dict(os.environ, C1EN_GPU=str(gpu))
    return subprocess.Popen(['node', 'motor/util/n-node.js', '/kaggle/temp/pi-trozo.esl', str(desde), str(hilos), str(MUESTRAS)],
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
# Una sola GPU con todo
t0 = time.time(); uno = lanzar(0, 0, HILOS).communicate()[0].strip(); t_uno = time.time() - t0
# Las dos a la vez, cada una con su mitad
t0 = time.time()
p0, p1 = lanzar(0, 0, HILOS // 2), lanzar(1, HILOS // 2, HILOS // 2)
s0, s1 = p0.communicate()[0].strip(), p1.communicate()[0].strip()
t_dos = time.time() - t0
print('una GPU :', uno)
print('GPU 0   :', s0)
print('GPU 1   :', s1)
c_uno = int(uno.split('|')[1]); c_dos = int(s0.split('|')[1]) + int(s1.split('|')[1])
print(f'\nπ ≈ {4 * c_uno / (HILOS * MUESTRAS):.8f}')
print('la suma de las dos mitades es EXACTAMENTE la cuenta de una GPU:', c_uno == c_dos)
print(f'tiempo de pared (con arranque de Node y NVRTC): una GPU {t_uno:.2f} s · dos GPU {t_dos:.2f} s')

## Qué queda comprobado aquí y no en otro sitio

En la máquina donde se escribió C1EÑ no había GPU. Allí se probó todo con un **simulador** (una `libcuda` y una `libnvrtc` falsas que compilan el CUDA C con g++ y recorren la cuadrícula en la CPU) y el CUDA C pasó por el front-end CUDA de clang hasta PTX, sin FMA. Lo que **solo** se puede comprobar en este cuaderno:

- que NVRTC (front-end EDG, no clang ni g++) acepta el texto generado;
- que las firmas de la Driver API escritas a mano en `motor/util/c1en-cuda.js` son las del driver real;
- que la aritmética de la T4 con `--fmad=false` da **los mismos bits** que la CPU (la celda de la suite con `--gpu`);
- los tiempos de verdad.

Si algo falla, la salida de la suite dice qué caso y con qué valores.